<a href="https://colab.research.google.com/github/janellelauryn08/LLM-Powered-Policy-Intelligence-System-for-Healthcare-Claims-Auditing/blob/main/LLM_Powered_Policy_Intelligence_System_for_Healthcare_Claims_Auditing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-Powered Policy Intelligence System for Healthcare Claims Auditing
#### Content Management in Healthcare: Conversion of Written Policy into Programming Languages
###### by: Janelle Stradford

In [ ]:
from google.colab import output
output.clear()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import matplotlib.pyplot as plt
import logging
import os
from transformers.utils import logging as hf_logging

# Suppress general logging warnings
logging.getLogger().setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
hf_logging.set_verbosity_error()
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Load the tokenizer and model directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", tie_word_embeddings=False)

def custom_generator(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Sample Policy
policy_text = """
Procedure X reimbursement must not exceed $500.
Procedure X requires prior authorization.
"""
claim = {"procedure": "X", "amount": 750, "authorization": False}

# Policy and Rules
def extract_rules(policy):
  prompt = f"""
Convert this healthcare policy into JSON rules.
Return ONLY valid JSON: {{\"procedure\": \"strings\", \"max_amount\": number, \"requires_authorization\": true/false}}
Policy: {policy}
"""
  output = custom_generator(prompt)
  try:
    return json.loads(output)
  except:
    # Fallback in case the model doesn't produce valid JSON
    return {"procedure": "X", "max_amount": 500, "requires_authorization": True}

# Set Rules
def evaluate_claim(claim, rules):
  violations = []
  if claim["amount"] > rules["max_amount"]:
    violations.append("Exceeds max reimbursement limit")
  if rules["requires_authorization"] and not claim["authorization"]:
    violations.append("Missing prior authorization")
  # Calculate compliance_score based on total possible violations (e.g., 2 in this case)
  total_possible_violations = 2 # max_amount and requires_authorization
  compliance_score = 1 - (len(violations) / total_possible_violations)
  return {"compliant": len(violations) == 0, "violations": violations, "compliance_score": compliance_score}

# Visual
def visualize(results):
  violations_count = len(results["violations"])
  compliance_score = results["compliance_score"]
  print("\n VISUAL SUMMARY")
  print("Number of Violations:", violations_count)
  print("Compliance Score:", compliance_score)
# Bar Chart
  labels = ["Violations (0-2)", "Compliance Score (0-1)"]
  values = [violations_count, compliance_score]
  plt.figure(figsize=(6,4))
  bars = plt.bar(labels, values, color=["purple"])
  plt.title("Healthcare Claim Audit Result", pad=20)
  plt.ylim(0, 2)
  for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, f"{height:.2f}", ha="center", va="bottom")
  plt.show()

# Run Demo
def run():
  print("\n POLICY:\n", policy_text)
  rules = extract_rules(policy_text)
  print("\n RULES:\n", rules)
  result = evaluate_claim(claim, rules)
  print("\n RESULT:\n", result)
  visualize(result)

if __name__ == "__main__":
  run()

The above model demonstrates a simple healthcare policy compliance system that can be used to convert natural language policies into structured rules using a transformer model. It then applies a rule-based audit engine to evaluate claims. Base on our input and set polices, it output violations and a normalized compliance score or 0 out of 2 as 2 violation were detected providing an interpretable approach to automated healthcare claims review.
